### Connection MYSQL

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [3]:
import pandas as pd

### AKM

In [4]:
akmQuery=text("""
WITH students AS (
    SELECT
        registers.id,
        registers.uuid,
        semester_code AS semester_student,
        registers.nim,
        prodi_registers.name AS prodi_student,
        id_jenis_keluar AS jenis_keluar_student
    FROM
        mahasiswa_registers AS registers
    JOIN 
        prodi AS prodi_registers ON registers.prodi_uuid = prodi_registers.uuid
    WHERE
        registers.deleted_at IS NULL 
        AND registers.is_external = 0 
        AND registers.is_archive = 0
),
akm AS (
    SELECT
        s.semester_student,
        s.nim,
        s.prodi_student,
        s.jenis_keluar_student,
        cuti.semester_start AS semester_cuti_start,
        keluar.semester_code AS semester_leave,
        keluar.jeniskeluar_code,
        kelas_laporan.semester_code AS semester_class_report,
		mbkm_activities.semester_code AS semester_mbkm
    FROM students s
    -- Join Cuti
    LEFT JOIN mahasiswa_cuti AS cuti 
        ON cuti.mahasiswa_register_uuid = s.uuid 
        AND cuti.deleted_at IS NULL 
        AND cuti.latest_status = 'approved'
    -- Join Keluar
    LEFT JOIN mahasiswa_keluar_details AS keluar_details 
        ON keluar_details.mahasiswa_register_uuid = s.uuid
    LEFT JOIN mahasiswa_keluar AS keluar 
        ON keluar_details.keluar_id = keluar.id
    -- Join Nilai & Kelas Laporan
    LEFT JOIN mahasiswa_nilais AS nilai 
        ON nilai.mahasiswa_register_id = s.id 
        AND nilai.deleted_at IS NULL
    LEFT JOIN kelas 
        ON nilai.kelas_id = kelas.id 
        AND kelas.deleted_at IS NULL
    LEFT JOIN kelas_laporan 
        ON kelas.kelas_laporan_uuid = kelas_laporan.uuid  
        AND kelas_laporan.deleted_at IS NULL
		-- Join MBKM
	LEFT JOIN mbkm_mahasiswa_applications AS mbkm_mahasiswa
		ON mbkm_mahasiswa.mahasiswa_register_uuid = s.uuid
	LEFT JOIN mahasiswa_activities AS mbkm_activities
		ON mbkm_mahasiswa.mahasiswa_activity_uuid = mbkm_activities.uuid
)
SELECT
    semester_student,
	nim,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_cuti_start,
	semester_mbkm
FROM
    akm
GROUP BY
    semester_student,
	nim,
    prodi_student,
    jenis_keluar_student,
    semester_class_report,
    semester_leave,
    jeniskeluar_code,
    semester_cuti_start,
	semester_mbkm
""")

In [5]:
akmDf = pd.read_sql(akmQuery, siakadEngine)

In [6]:
akmDf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14318 entries, 0 to 14317
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   semester_student       14318 non-null  object
 1   nim                    14318 non-null  object
 2   prodi_student          14318 non-null  object
 3   jenis_keluar_student   5572 non-null   object
 4   semester_class_report  13634 non-null  object
 5   semester_leave         5568 non-null   object
 6   jeniskeluar_code       5568 non-null   object
 7   semester_cuti_start    1294 non-null   object
 8   semester_mbkm          3878 non-null   object
dtypes: object(9)
memory usage: 1006.9+ KB


In [7]:
akmDf

,semester_student,nim,prodi_student,jenis_keluar_student,semester_class_report,semester_leave,jeniskeluar_code,semester_cuti_start,semester_mbkm
0,20181,1822006,Teknik Komputer,1,20182,20221,1,None,None
1,20181,1822006,Teknik Komputer,1,20181,20221,1,None,None
2,20181,1822006,Teknik Komputer,1,20191,20221,1,None,None
3,20181,1822006,Teknik Komputer,1,20192,20221,1,None,None
4,20181,1822006,Teknik Komputer,1,20201,20221,1,None,None
...,...,...,...,...,...,...,...,...,...
14313,20261,2631094,Perdagangan Internasional,None,None,None,None,None,None
14314,20261,2611281,Teknik Industri,None,None,None,None,None,None
14315,20261,2611282,Teknik Industri,None,None,None,None,None,None
14316,20261,2622059,Teknik Komputer,None,None,None,None,None,None


### Nilai Outlier

In [8]:
outlier = (akmDf['jenis_keluar_student'] != akmDf['jeniskeluar_code']) & (
    akmDf['jenis_keluar_student'].notna()
)

In [9]:
akmDf.loc[outlier, ['nim', 'prodi_student', 'jenis_keluar_student', 'jeniskeluar_code']].drop_duplicates()

,nim,prodi_student,jenis_keluar_student,jeniskeluar_code
25,1812009,Manajemen Rekayasa,6,None
4202,2124002,Matematika,6,None


In [10]:
akmClearDf = akmDf[~outlier].copy()

In [11]:
akmClearDf

,semester_student,nim,prodi_student,jenis_keluar_student,semester_class_report,semester_leave,jeniskeluar_code,semester_cuti_start,semester_mbkm
0,20181,1822006,Teknik Komputer,1,20182,20221,1,None,None
1,20181,1822006,Teknik Komputer,1,20181,20221,1,None,None
2,20181,1822006,Teknik Komputer,1,20191,20221,1,None,None
3,20181,1822006,Teknik Komputer,1,20192,20221,1,None,None
4,20181,1822006,Teknik Komputer,1,20201,20221,1,None,None
...,...,...,...,...,...,...,...,...,...
14313,20261,2631094,Perdagangan Internasional,None,None,None,None,None,None
14314,20261,2611281,Teknik Industri,None,None,None,None,None,None
14315,20261,2611282,Teknik Industri,None,None,None,None,None,None
14316,20261,2622059,Teknik Komputer,None,None,None,None,None,None


### Latest Semester Laporan

In [12]:
latestSemesterLaporan = akmClearDf[
    'semester_class_report'
    ].drop_duplicates().iloc[-1]

latestSemesterLaporan

'20252'

### Master Semester

In [13]:
semesterQuery = text("""
SELECT
    code, year_of_study
FROM
    semesters
WHERE
    year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
     year_of_study <= ( YEAR ( CURDATE())) AND
     semester != 3
""")

semesterDf = pd.read_sql(semesterQuery, siakadEngine)

In [14]:
semesterDf

,code,year_of_study
0,20221,2022
1,20222,2022
2,20231,2023
3,20232,2023
4,20241,2024
5,20242,2024
6,20251,2025
7,20252,2025
8,20261,2026
9,20262,2026


### Unique Valid Student

In [15]:
studentDf = akmClearDf.loc[
    (akmDf['semester_leave'].isna()) |
    (
        (akmDf['semester_leave'] >= semesterDf['code'].iloc[0]) &
        (akmDf['semester_leave'] <= semesterDf['code'].iloc[-1])
    ),
    ['nim', 'semester_student', 'prodi_student', 'semester_leave']
].drop_duplicates()

In [16]:
studentDf

,nim,semester_student,prodi_student,semester_leave
0,1822006,20181,Teknik Komputer,20221
21,1823024,20181,Desain Komunikasi Visual,20221
52,1811009,20181,Teknik Industri,20221
65,1811011,20181,Teknik Industri,20231
83,1811012,20181,Teknik Industri,20221
...,...,...,...,...
14313,2631094,20261,Perdagangan Internasional,None
14314,2611281,20261,Teknik Industri,None
14315,2611282,20261,Teknik Industri,None
14316,2622059,20261,Teknik Komputer,None


### Cross Join Semester x Student

In [17]:
studentMergeDf = semesterDf.merge(studentDf, how="cross")

In [18]:
studentMergeDf = studentMergeDf[
    (studentMergeDf["code"] >= studentMergeDf["semester_student"])
    &
    (
        (studentMergeDf["semester_leave"].isna())
        |
        (studentMergeDf["code"] <= studentMergeDf["semester_leave"])
    )
    &
    (studentMergeDf["code"] <= latestSemesterLaporan)
].drop(columns=["semester_leave", "semester_student"])

studentMergeDf['status'] = 'nonaktif'

In [19]:
studentMergeDf

,code,year_of_study,nim,prodi_student,status
0,20221,2022,1822006,Teknik Komputer,nonaktif
1,20221,2022,1823024,Desain Komunikasi Visual,nonaktif
2,20221,2022,1811009,Teknik Industri,nonaktif
3,20221,2022,1811011,Teknik Industri,nonaktif
4,20221,2022,1811012,Teknik Industri,nonaktif
...,...,...,...,...,...
24749,20252,2025,2511214,Teknik Industri,nonaktif
24750,20252,2025,2521071,Sistem Informasi,nonaktif
24751,20252,2025,2523067,Desain Komunikasi Visual,nonaktif
24752,20252,2025,2531055,Perdagangan Internasional,nonaktif


### Student Have Semester Laporan

In [20]:
studentSemesterLaporan = akmClearDf.loc[
    (akmClearDf['semester_class_report'].notna()) &
    (akmClearDf['semester_class_report'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_class_report']
].rename(columns={
    'semester_class_report': 'code'
}).drop_duplicates().copy()

In [21]:
studentSemesterLaporan['status_laporan'] = 'Aktif'

In [22]:
studentSemesterLaporan

,nim,code,status_laporan
9,1822006,20221,Aktif
60,1811009,20221,Aktif
73,1811011,20222,Aktif
74,1811011,20231,Aktif
91,1811012,20221,Aktif
...,...,...,...
13660,2531055,20251,Aktif
13661,2531055,20252,Aktif
13662,2521072,20252,Aktif
13902,2321054,20232,Aktif


### Student Have MBKM 

In [23]:
studentMbkm = akmClearDf.loc[
    (akmClearDf['semester_mbkm'].notna()) &
    (akmClearDf['semester_mbkm'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_mbkm']
].rename(columns={
    'semester_mbkm': 'code'
}).drop_duplicates().copy()

In [24]:
studentMbkm['status_mbkm'] = 'aktif'

In [25]:
studentMbkm

,nim,code,status_mbkm
857,1912006,20231,aktif
962,1923010,20231,aktif
984,1912009,20231,aktif
985,1912009,20232,aktif
1386,1923023,20252,aktif
...,...,...,...
12328,2421099,20261,aktif
12336,2421100,20261,aktif
12342,2421101,20261,aktif
12569,2412032,20241,aktif


### Student Have Cuti

In [26]:
studentCuti = akmClearDf.loc[
    (akmClearDf['semester_cuti_start'].notna()) &
    (akmClearDf['semester_cuti_start'] >= semesterDf['code'].iloc[0]),
    ['nim', 'semester_cuti_start']
].rename(columns={
    'semester_cuti_start': 'code'
}).drop_duplicates().copy()

In [27]:
studentCuti['status_cuti'] = 'cuti'

In [28]:
studentCuti

,nim,code,status_cuti
113,1823011,20221,cuti
277,1821012,20221,cuti
623,1912029,20221,cuti
697,1911002,20242,cuti
718,1911004,20221,cuti
...,...,...,...
12514,2411133,20251,cuti
12516,2411133,20252,cuti
12518,2422049,20251,cuti
12577,2431084,20251,cuti


### Student Graduate

In [29]:
studentGraduate = akmClearDf.loc[
    (akmClearDf['semester_leave'].notna()) &
    (akmClearDf['semester_leave'] >= semesterDf['code'].iloc[0]) &
    (akmClearDf['jeniskeluar_code'] == '1'),
    ['nim', 'semester_leave']
].rename(columns={
    'semester_leave': 'code'
}).drop_duplicates().copy()

In [30]:
studentGraduate['status_graduate'] = 'lulus';

In [31]:
studentGraduate

,nim,code,status_graduate
0,1822006,20221,lulus
52,1811009,20221,lulus
65,1811011,20231,lulus
83,1811012,20221,lulus
92,1823020,20222,lulus
...,...,...,...
6342,2121060,20251,lulus
8893,2212050,20242,lulus
8987,2223105,20251,lulus
11768,2411069,20242,lulus


### Final AKM

In [32]:
finalAkmDf = studentMergeDf.merge(
    studentSemesterLaporan, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentMbkm, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentCuti, on=['nim', 'code'], how='left'
)

finalAkmDf = finalAkmDf.merge(
    studentGraduate, on=['nim', 'code'], how='left'
)

In [33]:
finalAkmDf['status'] = (
    finalAkmDf.get('status_laporan')
    .combine_first(finalAkmDf.get('status_mbkm'))
    .combine_first(finalAkmDf.get('status_cuti'))
    .combine_first(finalAkmDf.get('status_graduate'))
    .combine_first(finalAkmDf.get('status'))   
)

cols_to_drop = [c for c in finalAkmDf.columns if c.startswith('status_') and c != 'status']
finalAkmDf = finalAkmDf.drop(columns=cols_to_drop, errors='ignore')

In [36]:
(
    finalAkmDf
    .rename(columns={
        'code': 'semester_code',
        'prodi_student': 'program_studi'
    })
    .to_csv('data/mahasiswa/akm.csv', index=False)
)